# 01 · Exploratory data analysis
Price distributions, missing data, correlations and location trends.

In [ ]:
import sys, json
sys.path.insert(0, "..")          # make `src` importable from /notebooks
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
raw = pd.read_csv(config.RAW_CSV, dtype={"zip_code": str})
clean = pd.read_csv(config.CLEAN_CSV, parse_dates=["sale_date"])
print(raw.shape, "->", clean.shape)
print(json.dumps(json.load(open(config.CLEANING_REPORT)), indent=2))

## Missing data (raw)

In [ ]:
raw.isna().mean().sort_values(ascending=False).head(8).mul(100).round(2).to_frame("% missing")

## Price distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
clean["sale_price"].plot.hist(bins=60, ax=ax[0], color="#2f6fe4"); ax[0].set_title("Sale price")
np.log10(clean["sale_price"]).plot.hist(bins=60, ax=ax[1], color="#8e6bd8"); ax[1].set_title("log10(sale price)")
plt.tight_layout(); plt.show()

## Price by property type and neighbourhood

In [ ]:
clean["price_per_sqft"] = clean["sale_price"] / clean["living_area"]
clean.boxplot(column="price_per_sqft", by="property_type", figsize=(7, 4)); plt.suptitle(""); plt.show()
clean.groupby("neighborhood")["price_per_sqft"].median().sort_values().plot.barh(figsize=(7, 4), color="#2f6fe4"); plt.show()

## Correlations

In [ ]:
cols = ["sale_price","living_area","bedrooms","bathrooms","lot_size","year_built","price_per_sqft"]
corr = clean[cols].corr()
fig, ax = plt.subplots(figsize=(6, 5)); im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right"); ax.set_yticks(range(len(cols)), cols)
fig.colorbar(im); plt.tight_layout(); plt.show()

## Trend over time

In [ ]:
clean.set_index("sale_date")["price_per_sqft"].resample("QE").median().plot(figsize=(8, 3.5), marker="o"); plt.ylabel("median $/sqft"); plt.show()